In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
tf.random.set_seed(42)

# 1. Generate diabetes-like dataset (500 records, 8 clinical features)
X, y = make_classification(n_samples=500, n_features=8, n_informative=5,
    n_redundant=1, n_classes=2, weights=[0.55, 0.45], random_state=42)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

results = []
histories = {}

def create_model(regularizer=None, dropout_rate=None):
    model = Sequential()
    model.add(Dense(256, activation='relu', kernel_regularizer=regularizer,
                    input_shape=(8,)))
    model.add(Dense(128, activation='relu', kernel_regularizer=regularizer))
    model.add(Dense(64, activation='relu', kernel_regularizer=regularizer))
    if dropout_rate is not None:
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

def train_model(name, model, epochs=100, callbacks=None):
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
        epochs=epochs, batch_size=16, verbose=0, callbacks=callbacks)
    train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    gap = train_acc - test_acc
    results.append({'Model': name, 'Train Accuracy': train_acc * 100,
        'Test Accuracy': test_acc * 100, 'Train-Test Gap': gap * 100,
        'Epochs': len(history.history['loss'])})
    histories[name] = history
    print(name)
    print("Training Accuracy:", round(train_acc * 100, 2), "%")
    print("Testing Accuracy:", round(test_acc * 100, 2), "%")
    print("Train-Test Gap:", round(gap * 100, 2), "%\n")
    return history

# 2. Baseline large MLP
baseline = create_model()
baseline_history = train_model("Baseline MLP", baseline, epochs=100)

# 3. L2 regularisation at three lambda values
for value in [0.001, 0.01, 0.1]:
    model = create_model(regularizer=l2(value))
    train_model(f"L2 Lambda={value}", model, epochs=100)

# 4. Dropout at three rates
for rate in [0.2, 0.3, 0.5]:
    model = create_model(dropout_rate=rate)
    train_model(f"Dropout={rate}", model, epochs=100)

# 5. Early stopping using validation loss
early_model = create_model()
early_stopping = EarlyStopping(monitor='val_loss', patience=10,
                               restore_best_weights=True)
train_model("Early Stopping", early_model, epochs=100,
            callbacks=[early_stopping])

# 6. Plot baseline overfitting
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(baseline_history.history['accuracy'], label='Training Accuracy')
plt.plot(baseline_history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Baseline MLP Accuracy'); plt.xlabel('Epochs')
plt.ylabel('Accuracy'); plt.legend(); plt.grid()
plt.subplot(1, 2, 2)
plt.plot(baseline_history.history['loss'], label='Training Loss')
plt.plot(baseline_history.history['val_loss'], label='Validation Loss')
plt.title('Baseline MLP Loss'); plt.xlabel('Epochs')
plt.ylabel('Loss'); plt.legend(); plt.grid()
plt.tight_layout(); plt.show()

# 7. Validation accuracy comparison
plt.figure(figsize=(12, 6))
for name, history in histories.items():
    plt.plot(history.history['val_accuracy'], label=name)
plt.title('Validation Accuracy Comparison'); plt.xlabel('Epochs')
plt.ylabel('Accuracy'); plt.legend(); plt.grid(); plt.show()

# 8. Final results table
df = pd.DataFrame(results)
print("\nFINAL MODEL COMPARISON")
print(df.round(2).to_string(index=False))

# 9. Test accuracy comparison
plt.figure(figsize=(12, 6))
plt.bar(df['Model'], df['Test Accuracy'])
plt.title('Test Accuracy of Different Models'); plt.xlabel('Model')
plt.ylabel('Test Accuracy (%)'); plt.xticks(rotation=45, ha='right')
plt.grid(axis='y'); plt.tight_layout(); plt.show()

# 10. Train-test gap comparison
plt.figure(figsize=(12, 6))
plt.bar(df['Model'], df['Train-Test Gap'])
plt.title('Train-Test Accuracy Gap Comparison'); plt.xlabel('Model')
plt.ylabel('Accuracy Gap (%)'); plt.xticks(rotation=45, ha='right')
plt.grid(axis='y'); plt.tight_layout(); plt.show()
